<div align="center">
<h1 style="background: linear-gradient(135deg, #8e44ad 0%, #9b59b6 100%); color: white; padding: 30px; border-radius: 20px; margin-bottom: 20px;">🔐 KRİPTOQRAFİYA KURSU</h1>
<h2 style="color: #2c3e50; font-size: 2.5em; margin: 20px 0;">Məşğələ 25</h2>
<h2 style="color: #e74c3c; font-size: 2em; margin: 20px 0;">TLS/SSL Protokolu</h2>
<h3 style="color: #7f8c8d; font-size: 1.5em; margin: 20px 0;">Təhlükəsiz Veb Əlaqələri</h3>
<br>
<div style="background-color: #9b59b6; color: white; padding: 15px; border-radius: 50px; display: inline-block; font-size: 1.2em;">⏱️ Məşğələ vaxtı: 2 saat</div>
<br><br>
<p><em>Hazırlanma tarixi: 2024</em></p>
</div>

## 📑 Mündəricat

1. [Məşğələnin məqsədləri](#məşğələnin-məqsədləri)
2. [İcra mühiti və laboratoriya müqaviləsi](#icra-mühiti-və-laboratoriya-müqaviləsi)
3. [TLS versiyalarının sərhədləri](#tls-versiyalarının-sərhədləri)
4. [Lokal CA və server sertifikatı](#lokal-ca-və-server-sertifikatı)
5. [Yoxlanan loopback TLS server/client](#yoxlanan-loopback-tls-serverclient)
6. [TLS 1.2 və TLS 1.3 handshake](#tls-12-və-tls-13-handshake)
7. [Cipher suites və record layer](#cipher-suites-və-record-layer)
8. [Wireshark analiz sərhədləri](#wireshark-analiz-sərhədləri)
9. [Tarixi hücumlar və cari nəticələr](#tarixi-hücumlar-və-cari-nəticələr)
10. [Təhlükəsiz Python kontekstləri](#təhlükəsiz-python-kontekstləri)
11. [İnteqrasiya edilmiş özünü-yoxlama](#inteqrasiya-edilmiş-özünü-yoxlama)
12. [Ev tapşırığı və mənbələr](#ev-tapşırığı-və-mənbələr)


## 🎯 Məşğələnin məqsədləri

Bu məşğələdən sonra tələbə:

- SSL 3.0, TLS 1.2 və TLS 1.3 sərhədlərini qarışdırmadan izah edir;
- lokal CA və `localhost` server sertifikatı yaradır;
- Python `ssl` ilə loopback server/client-i eyni clean run-da işlədir;
- chain və hostname yoxlamasını aktiv saxlayır və mənfi testlərlə təsdiqləyir;
- TLS 1.2 və TLS 1.3 cipher-suite semantikasını ayırır;
- 0-RTT replay və forward-secrecy məhdudiyyətlərini izah edir;
- record header müşahidəsini record protection reallaşdırması adlandırmır;
- tarixi hücumun protokol, versiya və implementation sərhədini göstərir.


## 📚 İcra mühiti və laboratoriya müqaviləsi

Notebook Python standard library (`ssl`, `socket`, `threading`, `tempfile`) və `cryptography` istifadə edir. Şəbəkə yalnız `127.0.0.1` loopback-dır. Sertifikat və encrypted private key yalnız Python `SSLContext.load_cert_chain()` file-path API-si üçün müvəqqəti qovluqda yaradılır və test bitən kimi silinir.

Xarici host, paket quraşdırılması, `input()`, daimi key faylı və deaktiv edilmiş certificate/hostname verification yoxdur.


In [ ]:
from importlib.metadata import PackageNotFoundError, version
from importlib.util import find_spec

if find_spec("cryptography") is None:
    raise RuntimeError("Bu notebook üçün 'cryptography' kitabxanası tələb olunur")

try:
    CRYPTOGRAPHY_VERSION = version("cryptography")
except PackageNotFoundError as exc:
    raise RuntimeError("'cryptography' paylanması tapılmadı") from exc

print(f"Məcburi asılılıq hazırdır: cryptography {CRYPTOGRAPHY_VERSION}")


## 🔧 Hazırlıq

İcra qaydası:

1. bütün hüceyrələri yuxarıdan aşağıya işlədin;
2. server yalnız OS-in seçdiyi ephemeral loopback portunu dinləyir;
3. client ayrıca verilmiş lokal CA-ya etibar edir və `localhost` SAN-ını yoxlayır;
4. TLS 1.2 və dəstəklənirsə TLS 1.3 ayrıca məcbur edilir;
5. yanlış hostname və yanlış trust anchor uğursuz olmalıdır;
6. bütün müvəqqəti fayllar context çıxışında silinir.


### 3.1 Python/OpenSSL imkanlarının yoxlanması

Python 3.11 `SSLContext.set_ciphers()` vasitəsilə TLS 1.3 cipher suites-i idarə etmir; bu metod TLS 1.2 və əvvəlki suite-lərə təsir edir. `get_ciphers()` isə mövcud TLS 1.3 suite-lərini də göstərə bilər. Notebook private `ssl._...` API-lərindən istifadə etmir.


In [ ]:
import ipaddress
import queue
import socket
import ssl
import sys
import tempfile
import threading
from datetime import datetime, timedelta, timezone
from pathlib import Path

import cryptography
from cryptography import x509
from cryptography.hazmat.primitives import hashes, serialization
from cryptography.hazmat.primitives.asymmetric import padding, rsa
from cryptography.x509.oid import ExtendedKeyUsageOID, NameOID

required_ssl_symbols = (
    ssl.SSLContext,
    ssl.PROTOCOL_TLS_CLIENT,
    ssl.PROTOCOL_TLS_SERVER,
    ssl.TLSVersion.TLSv1_2,
)
assert all(required_ssl_symbols)
print(f"Python {sys.version_info.major}.{sys.version_info.minor}; {ssl.OPENSSL_VERSION}")


### 3.2 Lokal və müvəqqəti iş sərhədi

`TemporaryDirectory(dir=".")` cari writable qovluqda unikal kataloq yaradır. Private key encrypted PKCS#8 kimi yalnız həmin kataloqda qısa müddət qalır. Parolun kodda olması production password management nümunəsi deyil; məqsəd plaintext key faylını normal nümunəyə çevirmədən Python-un file-path TLS API-sini sınaqdan keçirməkdir.


In [ ]:
LAB_NOW = datetime(2026, 7, 21, 12, 0, tzinfo=timezone.utc)
LAB_CONTRACT = {
    "external_network": False,
    "loopback_only": True,
    "persistent_key_files": False,
    "certificate_verification": True,
    "hostname_verification": True,
    "interactive_input": False,
}

assert LAB_NOW.utcoffset() == timedelta(0)
assert LAB_CONTRACT["loopback_only"]
assert LAB_CONTRACT["certificate_verification"]
assert LAB_CONTRACT["hostname_verification"]
assert not LAB_CONTRACT["external_network"]
assert not LAB_CONTRACT["persistent_key_files"]
print("TLS laboratoriya müqaviləsi təsdiqləndi")


## 📖 TLS versiyalarının sərhədləri

Qısa zaman xətti:

| Versiya | İl / əsas sənəd | Cari tədris statusu |
|---|---|---|
| SSL 2.0 | 1995 | RFC 6176 ilə qadağan edilib |
| SSL 3.0 | 1996 | RFC 7568 ilə deprecated edilib |
| TLS 1.0 | 1999, RFC 2246 | RFC 8996 ilə deprecated edilib |
| TLS 1.1 | 2006, RFC 4346 | RFC 8996 ilə deprecated edilib |
| TLS 1.2 | 2008, RFC 5246 | Legacy/interoperability profilində hələ rast gəlinir |
| TLS 1.3 | 2018 RFC 8446; 2026 RFC 9846 | Yeni protocol design üçün əsas seçim |

RFC 9846 TLS 1.3 versiya nömrəsini dəyişməyən, RFC 8446 ilə backward-compatible yeniləmədir.


### Versiya downgrade və tətbiq profili

TLS versiyasını yalnız “server ən böyüyünü seçir” kimi izah etmək kifayət deyil. Client və server supported versions, tətbiq profili və lokal siyasət üzrə razılaşır. TLS 1.0/1.1-ə fallback edilməməlidir. TLS 1.2 dəstəyi interop qərarı ola bilər; yeni protokollar üçün RFC 9852 TLS 1.3 tələbini irəli sürür.

Bu laboratoriya downgrade sınağı deyil: hər testdə minimum və maksimum versiya eyni qoyulur və negotiated version ayrıca assertion ilə yoxlanır.


## 📜 Lokal CA və server sertifikatı

Server leaf sertifikatı öz-imzalı deyil: onu notebook-un lokal root CA-sı buraxır. Client həmin root-u test trust anchor kimi ayrıca yükləyir. Leaf-də `localhost` DNS SAN-ı, `127.0.0.1` IP SAN-ı, `serverAuth`, uyğun Key Usage, SKI və AKI var.


### 4.1 Yaddaşda CA və TLS server materialı

Root private key diskə yazılmır. Server private key yalnız `SSLContext.load_cert_chain()` üçün encrypted formada müvəqqəti fayla çevrilir. Sertifikatların etibarlılıq vaxtı sabit laboratoriya vaxtını əhatə edir; real deployment müddəti ayrıca siyasətdir.


In [ ]:
def common_name(value):
    return x509.Name([x509.NameAttribute(NameOID.COMMON_NAME, value)])


def create_tls_fixture(root_label="AS25 Local Root CA"):
    root_key = rsa.generate_private_key(public_exponent=65537, key_size=2048)
    root_name = common_name(root_label)
    root_cert = (
        x509.CertificateBuilder()
        .subject_name(root_name)
        .issuer_name(root_name)
        .public_key(root_key.public_key())
        .serial_number(x509.random_serial_number())
        .not_valid_before(LAB_NOW - timedelta(days=1))
        .not_valid_after(LAB_NOW + timedelta(days=365))
        .add_extension(x509.BasicConstraints(ca=True, path_length=0), critical=True)
        .add_extension(
            x509.KeyUsage(
                digital_signature=False,
                content_commitment=False,
                key_encipherment=False,
                data_encipherment=False,
                key_agreement=False,
                key_cert_sign=True,
                crl_sign=True,
                encipher_only=False,
                decipher_only=False,
            ),
            critical=True,
        )
        .add_extension(x509.SubjectKeyIdentifier.from_public_key(root_key.public_key()), critical=False)
        .add_extension(x509.AuthorityKeyIdentifier.from_issuer_public_key(root_key.public_key()), critical=False)
        .sign(root_key, hashes.SHA256())
    )

    server_key = rsa.generate_private_key(public_exponent=65537, key_size=2048)
    server_cert = (
        x509.CertificateBuilder()
        .subject_name(common_name("localhost"))
        .issuer_name(root_cert.subject)
        .public_key(server_key.public_key())
        .serial_number(x509.random_serial_number())
        .not_valid_before(LAB_NOW - timedelta(days=1))
        .not_valid_after(LAB_NOW + timedelta(days=30))
        .add_extension(x509.BasicConstraints(ca=False, path_length=None), critical=True)
        .add_extension(
            x509.KeyUsage(
                digital_signature=True,
                content_commitment=False,
                key_encipherment=True,
                data_encipherment=False,
                key_agreement=False,
                key_cert_sign=False,
                crl_sign=False,
                encipher_only=False,
                decipher_only=False,
            ),
            critical=True,
        )
        .add_extension(x509.ExtendedKeyUsage([ExtendedKeyUsageOID.SERVER_AUTH]), critical=False)
        .add_extension(
            x509.SubjectAlternativeName(
                [
                    x509.DNSName("localhost"),
                    x509.IPAddress(ipaddress.ip_address("127.0.0.1")),
                ]
            ),
            critical=False,
        )
        .add_extension(x509.SubjectKeyIdentifier.from_public_key(server_key.public_key()), critical=False)
        .add_extension(
            x509.AuthorityKeyIdentifier.from_issuer_public_key(root_key.public_key()),
            critical=False,
        )
        .sign(root_key, hashes.SHA256())
    )
    return {
        "root_key": root_key,
        "root_cert": root_cert,
        "server_key": server_key,
        "server_cert": server_cert,
    }


tls_fixture = create_tls_fixture()
assert tls_fixture["server_cert"].issuer == tls_fixture["root_cert"].subject
print("Lokal root CA və localhost server leaf sertifikatı yaradıldı")


### 4.2 Sertifikat profilinin yoxlanması

Leaf imzasının root public key-i ilə yoxlanması chain/hostname validation-un yalnız bir hissəsidir. Aşağıdakı hüceyrə profil və imzanı yoxlayır; real TLS client yoxlaması isə loopback handshake zamanı OpenSSL tərəfindən aparılır.


In [ ]:
def inspect_server_certificate(fixture):
    root = fixture["root_cert"]
    leaf = fixture["server_cert"]
    root.public_key().verify(
        leaf.signature,
        leaf.tbs_certificate_bytes,
        padding.PKCS1v15(),
        leaf.signature_hash_algorithm,
    )
    san = leaf.extensions.get_extension_for_class(x509.SubjectAlternativeName).value
    eku = leaf.extensions.get_extension_for_class(x509.ExtendedKeyUsage).value
    usage = leaf.extensions.get_extension_for_class(x509.KeyUsage).value
    constraints = leaf.extensions.get_extension_for_class(x509.BasicConstraints).value
    return {
        "dns_names": san.get_values_for_type(x509.DNSName),
        "ip_addresses": [str(value) for value in san.get_values_for_type(x509.IPAddress)],
        "server_auth": ExtendedKeyUsageOID.SERVER_AUTH in eku,
        "digital_signature": usage.digital_signature,
        "is_ca": constraints.ca,
        "issuer_signature": True,
    }


server_certificate_profile = inspect_server_certificate(tls_fixture)
assert server_certificate_profile == {
    "dns_names": ["localhost"],
    "ip_addresses": ["127.0.0.1"],
    "server_auth": True,
    "digital_signature": True,
    "is_ca": False,
    "issuer_signature": True,
}
print(server_certificate_profile)


### ✍️ Çalışma 1: Sertifikat profili — 1 bal

`clientAuth` məqsədli ayrıca leaf sertifikatı yaradın və server leaf ilə EKU/KU fərqlərini assertions ilə göstərin.

**Qəbul meyarı:** hər iki leaf eyni root-dan buraxılır; server leaf-də `serverAuth`, client leaf-də `clientAuth` var; heç biri CA deyil; private material output-a verilmir.


## 🌐 Yoxlanan loopback TLS server/client

Server `127.0.0.1` üzərində port `0`-a bind edir; əməliyyat sistemi boş ephemeral port seçir. Client CA trust və hostname yoxlamasını aktiv saxlayır. Test thread-i yalnız bir bağlantı qəbul edir, `ping` alır və `pong` qaytarır.


### 5.1 Server context və müvəqqəti material

`PROTOCOL_TLS_SERVER` server rolunu, `PROTOCOL_TLS_CLIENT` client rolunu aydın ayırır. Hər testdə `minimum_version == maximum_version` qoyulduğu üçün negotiated version gözlənilən versiyadan fərqlənə bilməz. Production-da versiya intervalı tətbiq profilinə görə seçilir.


In [ ]:
def materialize_tls_fixture(directory, fixture, prefix, password):
    directory = Path(directory)
    paths = {
        "ca": directory / f"{prefix}-ca.pem",
        "cert": directory / f"{prefix}-server-cert.pem",
        "key": directory / f"{prefix}-server-key.pem",
    }
    paths["ca"].write_bytes(fixture["root_cert"].public_bytes(serialization.Encoding.PEM))
    paths["cert"].write_bytes(fixture["server_cert"].public_bytes(serialization.Encoding.PEM))
    paths["key"].write_bytes(
        fixture["server_key"].private_bytes(
            serialization.Encoding.PEM,
            serialization.PrivateFormat.PKCS8,
            serialization.BestAvailableEncryption(password.encode("utf-8")),
        )
    )
    assert b"BEGIN ENCRYPTED PRIVATE KEY" in paths["key"].read_bytes()
    return paths


def create_server_context(paths, tls_version, password):
    context = ssl.SSLContext(ssl.PROTOCOL_TLS_SERVER)
    context.minimum_version = tls_version
    context.maximum_version = tls_version
    context.load_cert_chain(paths["cert"], paths["key"], password=password)
    return context


def create_client_context(paths, tls_version):
    context = ssl.SSLContext(ssl.PROTOCOL_TLS_CLIENT)
    context.minimum_version = tls_version
    context.maximum_version = tls_version
    context.verify_mode = ssl.CERT_REQUIRED
    context.check_hostname = True
    context.load_verify_locations(cafile=paths["ca"])
    return context


assert create_server_context and create_client_context and materialize_tls_fixture
print("Server/client context factory-ləri hazırdır")


### 5.2 Bir bağlantılı loopback sınağı

Müsbət test həm server, həm client tərəfindən negotiated version və cipher-i qeyd edir. Mənfi testlərdə client konkret `SSLCertVerificationError` almalıdır; “işləməsi üçün” `CERT_NONE` və `check_hostname=False` istifadə edilmir.


In [ ]:
def run_loopback_exchange(server_context, client_context, server_hostname):
    listener = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    listener.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
    listener.bind(("127.0.0.1", 0))
    listener.listen(1)
    listener.settimeout(5)
    address = listener.getsockname()
    server_result = queue.Queue()

    def server_worker():
        try:
            raw_socket, _ = listener.accept()
            with raw_socket:
                with server_context.wrap_socket(raw_socket, server_side=True) as tls_socket:
                    request = tls_socket.recv(4)
                    tls_socket.sendall(b"pong")
                    server_result.put(
                        {
                            "server_version": tls_socket.version(),
                            "server_cipher": tls_socket.cipher()[0],
                            "request": request,
                        }
                    )
        except (ssl.SSLError, OSError) as exc:
            server_result.put({"server_error": type(exc).__name__})
        finally:
            listener.close()

    thread = threading.Thread(target=server_worker, daemon=True)
    thread.start()
    client_result = {}
    try:
        with socket.create_connection(address, timeout=5) as raw_socket:
            with client_context.wrap_socket(raw_socket, server_hostname=server_hostname) as tls_socket:
                tls_socket.sendall(b"ping")
                client_result = {
                    "accepted": True,
                    "reply": tls_socket.recv(4),
                    "client_version": tls_socket.version(),
                    "client_cipher": tls_socket.cipher()[0],
                }
    except ssl.SSLCertVerificationError as exc:
        client_result = {
            "accepted": False,
            "client_error": type(exc).__name__,
            "verify_code": exc.verify_code,
        }

    thread.join(timeout=5)
    assert not thread.is_alive()
    return {**client_result, **server_result.get(timeout=1)}


supported_tls_versions = [ssl.TLSVersion.TLSv1_2]
if ssl.HAS_TLSv1_3:
    supported_tls_versions.append(ssl.TLSVersion.TLSv1_3)

fixture_password = "AS25-temporary-key-password"
loopback_results = {}
foreign_fixture = create_tls_fixture("AS25 Foreign Root CA")

with tempfile.TemporaryDirectory(dir=".") as temporary_directory:
    temporary_path = Path(temporary_directory)
    fixture_paths = materialize_tls_fixture(
        temporary_path,
        tls_fixture,
        "trusted",
        fixture_password,
    )
    foreign_paths = materialize_tls_fixture(
        temporary_path,
        foreign_fixture,
        "foreign",
        fixture_password,
    )

    for tls_version in supported_tls_versions:
        expected_name = "TLSv1.3" if tls_version == ssl.TLSVersion.TLSv1_3 else "TLSv1.2"
        result = run_loopback_exchange(
            create_server_context(fixture_paths, tls_version, fixture_password),
            create_client_context(fixture_paths, tls_version),
            "localhost",
        )
        assert result["accepted"] and result["reply"] == b"pong"
        assert result["request"] == b"ping"
        assert result["client_version"] == result["server_version"] == expected_name
        loopback_results[expected_name] = result

    negative_version = supported_tls_versions[-1]
    wrong_hostname_result = run_loopback_exchange(
        create_server_context(fixture_paths, negative_version, fixture_password),
        create_client_context(fixture_paths, negative_version),
        "wrong.example.test",
    )
    wrong_trust_result = run_loopback_exchange(
        create_server_context(fixture_paths, negative_version, fixture_password),
        create_client_context(foreign_paths, negative_version),
        "localhost",
    )

    assert not wrong_hostname_result["accepted"]
    assert wrong_hostname_result["client_error"] == "SSLCertVerificationError"
    assert not wrong_trust_result["accepted"]
    assert wrong_trust_result["client_error"] == "SSLCertVerificationError"

tls_temporary_cleanup = not temporary_path.exists()
assert tls_temporary_cleanup
print(
    {
        "positive_versions": sorted(loopback_results),
        "wrong_hostname_rejected": True,
        "wrong_trust_rejected": True,
        "temporary_files_removed": tls_temporary_cleanup,
    }
)


### ✍️ Çalışma 2: TLS server/client — 1 bal

Loopback fixture-i `127.0.0.1` SAN-ı ilə hostname əvəzinə IP identifier üzrə yoxlayın və ayrıca `expired certificate` mənfi testi əlavə edin.

**Qəbul meyarı:** verification heç vaxt söndürülmür; IP testi qəbul edilir; expired leaf `SSLCertVerificationError` ilə rədd edilir; thread və müvəqqəti fayllar sonda qalmır.


## 🤝 TLS 1.2 və TLS 1.3 handshake

Handshake versiyanı və kriptoqrafik parametrləri razılaşdırır, endpoint authentication-u tətbiq edir, shared traffic secrets yaradır və transcript üzərindən Finished doğrulaması aparır. Mesaj axını tətbiq olunan authentication, PSK/resumption, client auth və HelloRetryRequest kimi seçimlərə görə dəyişə bilər.


### 6.1 TLS 1.2 handshake sərhədi

RFC 5246 cipher suite adətən key exchange, authentication, bulk encryption, MAC və PRF seçimini birlikdə ifadə edir. Məsələn, `TLS_ECDHE_RSA_WITH_AES_128_GCM_SHA256`:

- ECDHE — ephemeral key exchange;
- RSA — server certificate/signature authentication;
- AES-128-GCM — AEAD record protection;
- SHA-256 — TLS 1.2 PRF ilə əlaqəli hash.

TLS 1.2 həm legacy CBC, həm də AEAD suite-lərə malikdir; buna görə bütün TLS 1.2 record-larını eyni “MAC-then-encrypt” pipeline ilə təsvir etmək olmaz.


### 6.2 TLS 1.3 handshake sərhədi

RFC 9846-da cipher suite yalnız record-protection AEAD alqoritmini və HKDF/transcript üçün hash-i seçir. Key exchange group və signature algorithm ayrıca extensions ilə razılaşdırılır. Static RSA/DH suite-ləri, record compression və legacy alqoritmlər çıxarılıb; ServerHello-dan sonrakı handshake mesajları şifrlənir.

Full (EC)DHE handshake keçmiş sessiya traffic-i üçün forward secrecy verir. PSK-only resumption eyni zəmanəti verməyə bilər; BCP 195 `psk_dhe_ke` seçimini tövsiyə edir.


### 6.3 TLS 1.2 və TLS 1.3 müqayisəsi

| Mövzu | TLS 1.2 | TLS 1.3 |
|---|---|---|
| Cipher suite | Key exchange/auth/record/PRF bir adda ola bilər | AEAD + hash; key exchange və signature ayrıdır |
| Full handshake latency | Tipik axında application data-dan əvvəl daha çox flight | Tipik full handshake 1-RTT |
| Record protection | Suite-dən asılı CBC və ya AEAD | AEAD |
| Compression | Protokolda mövcud idi; təhlükəsiz profil null seçir | Record compression çıxarılıb |
| Static RSA key exchange | Mövcud idi, PFS vermirdi | Çıxarılıb |
| 0-RTT early data | Yoxdur | Yalnız uyğun PSK/resumption profilində, zəif replay/FS xassələri ilə |

Bu cədvəl bütün extension və resumption variasiyalarını əhatə etmir.


### 6.4 Forward secrecy

Forward secrecy uzunmüddətli authentication key sonradan oğurlandıqda əvvəlki session traffic-in dərhal açılmamasını hədəfləyir. Bu xassə üçün ephemeral secret-lərin düzgün yaradılması və silinməsi, transcript authentication və uyğun key exchange lazımdır.

“TLS 1.3 həmişə PFS verir” həddən artıq ümumidir: 0-RTT data PSK-dan yaranan early traffic key ilə qorunur və forward secret deyil; PSK-only resumption da yeni (EC)DHE olmadan fərqli sərhədə malikdir.


### 6.5 0-RTT early data

TLS 1.3 0-RTT yalnız client və server əvvəlcədən PSK paylaşdıqda ilk flight-da application data göndərməyə imkan verir. RFC 9846 üzrə:

- early data forward secret deyil;
- bağlantılar arasında non-replay zəmanəti yoxdur;
- idempotent olmaq minimum şərtdir, lakin həmişə kifayət deyil;
- server anti-replay state/freshness siyasəti, application isə replay-safe message profile müəyyən etməlidir;
- implementation açıq şəkildə aktiv etməyibsə, 0-RTT istifadə olunmamalıdır.

Python 3.11 `ssl` API-si early data göndərmə/qəbul etməni təqdim etmir; notebook bunu simulyasiya edib “test edildi” demir.


### 6.6 Cipher suites

Suite adını oxuyarkən əvvəlcə negotiated TLS versiyasını müəyyən edin. Eyni `SSLContext.get_ciphers()` siyahısında TLS 1.2 və TLS 1.3 adları görünə bilər, lakin semantikası fərqlidir. `SSLContext.set_ciphers()` ilə TLS 1.3 suite-lərini dəyişdirdiyinizi güman etməyin.


In [ ]:
def cipher_suite_inventory():
    context = ssl.SSLContext(ssl.PROTOCOL_TLS_CLIENT)
    before = context.get_ciphers()
    tls13_before = sorted({item["name"] for item in before if item["protocol"] == "TLSv1.3"})
    tls12_before = sorted({item["name"] for item in before if item["protocol"] == "TLSv1.2"})

    preferred_tls12 = "ECDHE-RSA-AES128-GCM-SHA256"
    if preferred_tls12 in tls12_before:
        context.set_ciphers(preferred_tls12)
    after = context.get_ciphers()
    tls13_after = sorted({item["name"] for item in after if item["protocol"] == "TLSv1.3"})

    assert tls12_before
    if ssl.HAS_TLSv1_3:
        assert tls13_before and tls13_after == tls13_before
        assert all(name.startswith("TLS_") for name in tls13_before)
    return {
        "tls12_count_before_filter": len(tls12_before),
        "tls13_names": tls13_before,
        "set_ciphers_left_tls13_unchanged": tls13_after == tls13_before,
    }


cipher_suite_result = cipher_suite_inventory()
print(cipher_suite_result)


### ✍️ Çalışma 3: Handshake siyasəti — 1 bal

Loopback nəticələrində TLS 1.2 və TLS 1.3 cipher adlarını parçalayın və hər hissənin semantikasını yazın.

**Qəbul meyarı:** TLS 1.2-də ECDHE/RSA/AEAD/hash rolları, TLS 1.3-də isə yalnız AEAD+hash suite semantikası göstərilir; negotiated version nəticədən oxunur; `set_ciphers()` TLS 1.3 nəzarəti kimi təqdim edilmir.


## 📦 Record layer

Record layer handshake-dən alınmış traffic secret-lər əsasında məxfilik, bütövlük və sıra/non-replay xassələrini təmin etməyə çalışır. TLS 1.2 protection negotiated suite-dən asılıdır; TLS 1.3 yalnız AEAD istifadə edir və application data daxil olmaqla qorunan record-larda outer content type-i fərqli təqdim edə bilər.


### 7.1 Record header müşahidəsi

TLSPlaintext/TLSCiphertext wire header-i 1 bayt content type, 2 bayt legacy record version və 2 bayt fragment length sahəsindən ibarətdir. TLS 1.3 qorunan record üçün `legacy_record_version` dəyərinin `0x0303` olması negotiated version-un TLS 1.2 olduğunu göstərmir.

Aşağıdakı parser yalnız 5-bayt header-i oxuyur; encryption, authentication, nonce və key schedule reallaşdırmır və TLS implementation deyil.


In [ ]:
TLS_CONTENT_TYPES = {
    20: "change_cipher_spec",
    21: "alert",
    22: "handshake",
    23: "application_data",
}


def parse_tls_record_header(record):
    if len(record) < 5:
        raise ValueError("TLS record header üçün ən azı 5 bayt lazımdır")
    content_type = record[0]
    legacy_version = int.from_bytes(record[1:3], "big")
    fragment_length = int.from_bytes(record[3:5], "big")
    if len(record) != 5 + fragment_length:
        raise ValueError("Record uzunluğu header ilə uyğun deyil")
    return {
        "content_type": TLS_CONTENT_TYPES.get(content_type, "unknown"),
        "legacy_record_version": f"0x{legacy_version:04x}",
        "fragment_length": fragment_length,
    }


sample_tls13_ciphertext_record = b"\x17\x03\x03\x00\x04" + b"\x00\x01\x02\x03"
record_header_result = parse_tls_record_header(sample_tls13_ciphertext_record)
assert record_header_result == {
    "content_type": "application_data",
    "legacy_record_version": "0x0303",
    "fragment_length": 4,
}

truncated_record_rejected = False
try:
    parse_tls_record_header(b"\x17\x03\x03\x00\x04\x00")
except ValueError:
    truncated_record_rejected = True

assert truncated_record_rejected
print({**record_header_result, "truncated_rejected": truncated_record_rejected})


### ✍️ Çalışma 4: Record header — 1 bal

Handshake, alert və application-data header nümunələri yaradın; yanlış length və naməlum content-type halları üçün test yazın.

**Qəbul meyarı:** parser yalnız header metadata-sı qaytarır; negotiated TLS version-u `legacy_record_version`-dan çıxarmır; encryption/MAC/AEAD etdiyini iddia etmir.


## 🔍 Wireshark analiz sərhədləri

Paket capture şəbəkədə görünən record və handshake metadata-sını müşahidə edə bilər. TLS 1.3-də ServerHello-dan sonrakı handshake mesajları şifrlənir. Application plaintext-i görmək üçün endpoint-dən alınmış session secrets və uyğun decryption workflow lazımdır; private server key təkbaşına modern (EC)DHE sessiyalarını açmır.


### 8.1 Filtrlər və sübut səviyyəsi

Wireshark filter nəticəsi “sertifikat etibarlıdır” sübutu deyil. Capture aşağıdakı suallara cavab verə bilər: hansı TCP flow, hansı görünən handshake/record tipi, hansı negotiated version/cipher metadata-sı və alert baş verib? Chain, hostname və application semantics endpoint/verifier sübutu ilə tamamlanmalıdır.

Session-key log həssas materialdır: yalnız icazəli laboratoriyada yaradılmalı, repozitoriyaya əlavə edilməməli və iş bitdikdə təhlükəsiz silinməlidir.


In [ ]:
def wireshark_filter_catalog():
    filters = {
        "tls_records": "tls",
        "client_hello": "tls.handshake.type == 1",
        "server_hello": "tls.handshake.type == 2",
        "alerts": "tls.alert_message",
        "loopback_tcp": "ip.addr == 127.0.0.1 && tcp",
    }
    assert all(isinstance(value, str) and value for value in filters.values())
    return filters


wireshark_filters = wireshark_filter_catalog()
print(wireshark_filters)


### ✍️ Çalışma 5: Capture planı — 1 bal

İcazəli lokal loopback sınağı üçün capture planı yazın: başlanğıc/bitiş, filter, gözlənilən görünən sahələr, görünməyən məlumat və təmizləmə addımı.

**Qəbul meyarı:** real capture məcburi deyil; plan TLS 1.3 encrypted handshake sərhədini, session-secret həssaslığını və capture nəticəsinin certificate validation əvəzi olmadığını göstərir.


## ⚠️ Tarixi hücumlar və cari nəticələr

Tarixi hücumu bug adı kimi əzbərləmək əvəzinə dörd sahə ilə təsnif edin: hədəf versiya/feature, ön şərt, əldə olunan nəticə və düzəliş sərhədi. “TLS sındırıldı” kimi ümumi cümlə konkret protokol və implementation problemini gizlədir.


### 9.1 POODLE

Əsas POODLE hücumu SSL 3.0 CBC padding davranışından və downgrade imkanından istifadə edən padding oracle-dır; SSL 2.0 hücumu deyil. TLS implementation-larında oxşar padding yoxlama qüsurları ayrıca “POODLE-TLS” adlandırılıb. Praktik nəticə SSL 3.0-u söndürmək, təhlükəsiz fallback siyasəti və düzgün authenticated encryption profilidir.


### 9.2 BEAST

BEAST browser kontekstində TLS 1.0 CBC record-larının əvvəlki ciphertext block-dan yaranan predictable IV xüsusiyyətini və chosen-plaintext imkanını hədəfləyirdi. TLS 1.1+ explicit IV istifadə edir; müasir nəticə deprecated TLS 1.0-a qayıtmamaq və cari AEAD profillərinə üstünlük verməkdir.


### 9.3 CRIME və BREACH fərqi

CRIME TLS-level compression ilə secret və attacker-controlled plaintext-i eyni compression kontekstində müşahidə etməyə əsaslanır. TLS compression-u söndürmək CRIME sərhədinə cavabdır. BREACH isə HTTP response compression kimi application-layer compression-u hədəfləyir; TLS compression-un bağlı olması onu avtomatik həll etmir.


### 9.4 Heartbleed

Heartbleed OpenSSL heartbeat extension implementation-ında bounds-check qüsuru idi və proses yaddaşından məlumat sızdıra bilirdi. Bu, TLS cipher-in kriptoanalizi deyildi. Patch-dən əlavə kompromis ehtimalına görə key/certificate replacement, credential rotation və incident analysis tələb oluna bilərdi.


### 9.5 FREAK və Logjam

FREAK bəzi implementation-larda export-grade RSA seçiminə downgrade etməyə imkan verirdi. Logjam export-grade və zəif/ortaq finite-field DH qruplarını hədəfləyirdi. Nəticə yalnız suite adını dəyişmək deyil: legacy/export seçimlərini söndürmək, güclü qruplar və forward-secret profillər seçmək, negotiation və downgrade davranışını test etməkdir.


In [ ]:
def tls_attack_catalog():
    attacks = {
        "POODLE": {
            "boundary": "SSL 3.0 CBC padding oracle; TLS variants were implementation bugs",
            "not": "SSL 2.0 attack",
        },
        "BEAST": {
            "boundary": "TLS 1.0 CBC predictable-IV browser scenario",
            "not": "generic AEAD break",
        },
        "CRIME": {
            "boundary": "TLS-level compression side channel",
            "not": "the same layer as BREACH",
        },
        "Heartbleed": {
            "boundary": "OpenSSL heartbeat bounds-check bug",
            "not": "TLS cipher cryptanalysis",
        },
        "FREAK": {
            "boundary": "export-RSA downgrade/implementation behavior",
            "not": "RSA private-key recovery from every modern TLS session",
        },
        "Logjam": {
            "boundary": "export/weak finite-field DH groups",
            "not": "ECDHE break",
        },
    }
    assert attacks["POODLE"]["boundary"].startswith("SSL 3.0")
    assert "implementation" in attacks["Heartbleed"]["boundary"] or "OpenSSL" in attacks["Heartbleed"]["boundary"]
    return attacks


tls_attacks = tls_attack_catalog()
print({name: details["boundary"] for name, details in tls_attacks.items()})


### ✍️ Çalışma 6: Hücum sərhədi — 1 bal

Bir tarixi TLS hücumu üçün `version/feature → prerequisite → attacker result → mitigation → non-result` cədvəli hazırlayın.

**Qəbul meyarı:** ən azı bir primary standard/BCP və bir texniki mənbə göstərilir; protokol qüsuru implementation bug-dan ayrılır; düzəliş “ən son TLS istifadə et” cümləsi ilə məhdudlaşmır.


## 🔒 Təhlükəsiz Python kontekstləri

Client context üçün `PROTOCOL_TLS_CLIENT`, `CERT_REQUIRED`, `check_hostname=True` və uyğun trust store birlikdə işləyir. Server context üçün `PROTOCOL_TLS_SERVER` istifadə olunur; server-side `check_hostname` anlayışı client identifier yoxlamasına aid deyil. Minimum versiya və suite siyasəti deployment profilinə görə seçilir.


In [ ]:
def inspect_context_policy(server_context, client_context, tls_version):
    return {
        "server_role": server_context.protocol == ssl.PROTOCOL_TLS_SERVER,
        "client_role": client_context.protocol == ssl.PROTOCOL_TLS_CLIENT,
        "client_cert_required": client_context.verify_mode == ssl.CERT_REQUIRED,
        "client_hostname_check": client_context.check_hostname,
        "server_minimum": server_context.minimum_version == tls_version,
        "client_minimum": client_context.minimum_version == tls_version,
        "exact_version_test": (
            server_context.maximum_version == tls_version
            and client_context.maximum_version == tls_version
        ),
    }


policy_version = supported_tls_versions[-1]
with tempfile.TemporaryDirectory(dir=".") as policy_directory:
    policy_path = Path(policy_directory)
    policy_paths = materialize_tls_fixture(
        policy_path,
        tls_fixture,
        "policy",
        fixture_password,
    )
    secure_context_checks = inspect_context_policy(
        create_server_context(policy_paths, policy_version, fixture_password),
        create_client_context(policy_paths, policy_version),
        policy_version,
    )

assert not policy_path.exists()
assert all(secure_context_checks.values())
print(secure_context_checks)


### ✍️ Çalışma 7: Context review — 1 bal

Verilmiş client/server context konfiqurasiyasında rol, trust store, hostname, minimum version və cipher-policy səhvlərini tapmaq üçün checklist hazırlayın.

**Qəbul meyarı:** `CERT_NONE` və `check_hostname=False` “localhost üçün təhlükəsizdir” kimi qəbul edilmir; server və client rolları ayrılır; TLS 1.3 suite nəzarəti `set_ciphers()`-a aid edilmir.


## 🖥️ İnteqrasiya edilmiş özünü-yoxlama

Son hüceyrə bütün məcburi sübutları birləşdirir. `pass` yalnız bu lokal fixture-in chain/hostname/version/echo və sənədləşdirilmiş metadata testlərinə aiddir; public Web PKI, internet host və ya Wireshark decryption end-to-end yoxlanmır.


In [ ]:
def run_tls_lab_self_check():
    checks = {
        "certificate_profile": all(
            (
                server_certificate_profile["server_auth"],
                server_certificate_profile["digital_signature"],
                not server_certificate_profile["is_ca"],
            )
        ),
        "tls12_loopback": loopback_results["TLSv1.2"]["accepted"],
        "tls13_loopback_if_supported": (
            not ssl.HAS_TLSv1_3 or loopback_results["TLSv1.3"]["accepted"]
        ),
        "hostname_rejection": not wrong_hostname_result["accepted"],
        "trust_anchor_rejection": not wrong_trust_result["accepted"],
        "temporary_cleanup": tls_temporary_cleanup and not policy_path.exists(),
        "tls13_cipher_boundary": cipher_suite_result["set_ciphers_left_tls13_unchanged"],
        "record_header_parser": record_header_result["fragment_length"] == 4,
        "attack_boundaries": len(tls_attacks) == 6,
        "context_policy": all(secure_context_checks.values()),
    }
    assert all(checks.values())
    return checks


tls_lab_checks = run_tls_lab_self_check()
print({"passed": sum(tls_lab_checks.values()), "total": len(tls_lab_checks)})


## 🏠 Ev tapşırığı — 10 bal

Loopback laboratoriyasını qarşılıqlı TLS (mTLS) ilə genişləndirin.

| Meyar | Bal |
|---|---:|
| Root, server və client certificate profilləri düzgün ayrılıb | 2 |
| Server client CA trust-u və `CERT_REQUIRED` tətbiq edir | 2 |
| Client server chain və `localhost` SAN-ını yoxlayır | 1 |
| Müsbət mTLS echo testi | 1 |
| Wrong server name, wrong client CA və missing client certificate mənfi testləri | 2 |
| Müvəqqəti encrypted key faylları təmizlənir | 1 |
| Təhdid modeli və production-a keçid sərhədi sənədləşdirilir | 1 |

**Məhdudiyyətlər:** yalnız loopback; verification söndürülmür; `input()`, xarici host, daimi key və broad `except Exception` yoxdur.


## 📌 Yekun

- TLS 1.2 və TLS 1.3 cipher-suite və record semantikası eyni deyil.
- TLS client chain və hostname yoxlamasını birlikdə aparmalıdır.
- `set_ciphers()` Python 3.11-də TLS 1.3 suite siyasətini idarə etmir.
- 0-RTT replay-safe application profile tələb edir və early data forward secret deyil.
- TLS 1.3-də `legacy_record_version=0x0303` negotiated TLS 1.2 demək deyil.
- POODLE əsasən SSL 3.0, Heartbleed isə OpenSSL implementation sərhədidir.
- Local success production TLS siyasəti, public trust və revocation-u avtomatik təsdiqləmir.


## 📚 Əlavə resurslar

- RFC 9846 — The Transport Layer Security (TLS) Protocol Version 1.3
- RFC 8446 — TLS 1.3-ün ilkin spesifikasiyası; RFC 9846 ilə əvəz edilib
- RFC 5246 — The Transport Layer Security (TLS) Protocol Version 1.2
- RFC 9325 / BCP 195 — Recommendations for Secure Use of TLS and DTLS
- RFC 8996 — Deprecating TLS 1.0 and TLS 1.1
- RFC 7568 — Deprecating Secure Sockets Layer Version 3.0
- RFC 7457 — Summarizing Known Attacks on TLS and DTLS
- Python 3.11 `ssl` documentation — contexts, verification, TLS 1.3 API sərhədləri
- `cryptography` X.509 documentation — certificate builder və extensions

> BCP və kitabxana imkanları zamanla dəyişə bilər; production profili üçün tətbiq olunan cari standart, platforma versiyası və root policy ayrıca yoxlanmalıdır.
